# Transformers

Many models need preprocessed features, e.g. scaled features. Then the
preprocessing has to run in the target language too. This example ports a
`StandardScaler` to JavaScript with
[sklearn-porter](https://github.com/nok/sklearn-porter) and runs it with
Node.js.

Running the ported transformer requires `node`.

## Train a transformer

The features of the wine dataset have very different scales, e.g. the
alcohol (about 13) and the proline (about 1000):

In [1]:
import numpy as np
from sklearn.datasets import load_wine
from sklearn.preprocessing import StandardScaler

np.set_printoptions(precision=3, suppress=True)

X, y = load_wine(return_X_y=True)
scaler = StandardScaler()
scaler.fit(X)

print(X[:2])

[[  14.23    1.71    2.43   15.6   127.      2.8     3.06    0.28    2.29
     5.64    1.04    3.92 1065.  ]
 [  13.2     1.78    2.14   11.2   100.      2.65    2.76    0.26    1.28
     4.38    1.05    3.4  1050.  ]]


## Port the transformer

The class `Porter` (and the functions `port`, `save`, `make` and `test`)
accepts transformers like classifiers. Save the ported scaler:

In [2]:
from sklearn_porter import Porter

porter = Porter(scaler, language='js')
src_path = porter.save(directory='/tmp')

print(src_path)

/tmp/StandardScaler.js


## Run the ported transformer

The ported transformer prints the transformed features of each sample in
one line, separated by spaces:

In [3]:
!node /tmp/StandardScaler.js 14.23 1.71 2.43 15.6 127 2.8 3.06 0.28 2.29 5.64 1.04 3.92 1065

1.5186125409891542 -0.562249798328623 0.23205254099473993 -1.1695931750229027 1.9139052175708111 0.8089973946320399 1.0348189581307379 -0.6595631143050651 1.2248839840604513 0.2517168498188532 0.3621772757786129 1.8479195665066535 1.013008926747691


`transform` runs the ported transformer from Python, like `transform` of
scikit-learn:

In [4]:
print(porter.transform(X[:2]))
print(scaler.transform(X[:2]))

[[ 1.519 -0.562  0.232 -1.17   1.914  0.809  1.035 -0.66   1.225  0.252
   0.362  1.848  1.013]
 [ 0.246 -0.499 -0.828 -2.491  0.018  0.569  0.734 -0.821 -0.545 -0.293
   0.406  1.113  0.965]]
[[ 1.519 -0.562  0.232 -1.17   1.914  0.809  1.035 -0.66   1.225  0.252
   0.362  1.848  1.013]
 [ 0.246 -0.499 -0.828 -2.491  0.018  0.569  0.734 -0.821 -0.545 -0.293
   0.406  1.113  0.965]]


## Check the ported transformer

The score is the fraction of samples whose transformed features match
those of scikit-learn:

In [5]:
print(porter.test(X))

1.0


## Combine transformers

A `ColumnTransformer` (or a `FeatureUnion`) combines several transformers,
also transformers which change the number of features. Here the first 12
features are scaled and the proline is split into 3 bins (one-hot encoded),
so a sample has 15 transformed features:

In [6]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import KBinsDiscretizer

columns = ColumnTransformer(
    [
        ('scale', StandardScaler(), slice(0, 12)),
        ('bins', KBinsDiscretizer(n_bins=3, encode='onehot-dense'), [12]),
    ]
)
columns.fit(X)

porter = Porter(columns, language='js')

print(porter.transform(X[:2]))
print(porter.test(X))

[[ 1.519 -0.562  0.232 -1.17   1.914  0.809  1.035 -0.66   1.225  0.252
   0.362  1.848  0.     0.     1.   ]
 [ 0.246 -0.499 -0.828 -2.491  0.018  0.569  0.734 -0.821 -0.545 -0.293
   0.406  1.113  0.     0.     1.   ]]
1.0


The [README](https://github.com/nok/sklearn-porter#transformers) lists all
supported transformers.

## Next steps

- [Pipelines](pipelines.ipynb): port the preprocessing and the classifier
  of a pipeline to one source file.